# GBRAS timing/numerics benchmark — NOT an architecture result

One approved G4 boot; shared runtime/Lightning, real GBRAS128 recipe and60da.
Fresh sequential processes: FP32/NCHW, BF16/NCHW, BF16/channels-last;
optional FP32/channels-last only with reserve for winner validation.
25warm +100measured steps;200matched updates; no per-step GPU synchronization.
Frozen gates: loss windows50, gap <=max(.02,5% FP32 mean), same-weight
inference probability p99 gap <=.01, all finite. Both trained checkpoints
are compared with FP32 inference. Source helper stores gates and all evidence.
Winner gets one full real60da validation/checkpoint overhead measurement.
Internal deadline40min from first cell; trial8min/overhead10min limits; submit45/backstop60min.
All JSON/checkpoints are marked benchmark, not a result; no full-run evaluation.
PC only: CVDL_BENCH_SMOKE=1 runs tiny CPU fixtures, compileoff, no hardware timings.


In [ ]:
import time
BENCH_STARTED = time.monotonic()
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "cvdl").is_dir())
sys.path.insert(0, str(ROOT))
from cvdl.settings import load

settings = load(
    name="gbras-speed-bench",
    front_end="srm",
    tile_size=512,
    train_tile_size=256,
    train_grid="overlap_last",
    batch_size=128,
    paired_batches=False,
    tiles_per_epoch=180000,
    post_fit_eval="none",
    model='gbras',
    norm='batch',
    clip_t=3.0,
    lr=0.0003,
    warmup_epochs=1,
    grad_clip=1.0,
    lr_drops=((5, 0.3), (20, 0.1)),
    epochs=30,
    train_yaml='configs/train-k1to4.yaml',
    data='otf',
    source='decode',
    source_mix={'raise': 0.4, 'lsdir': 0.2, 'alaska': 0.4},
    evaluation_checkpoint='best',
    checkpoint_validation='train_matched',
    validation_recipe='60da4391043e8ce48ff2',
    val_cap_raise=5000,
    val_cap_lsdir=1500,
    extra_tests='test-bpp1,source-panels',
)
settings.apply_runtime_environment()
RUN_NAME = settings.run_name


In [ ]:
import json
import os
from cvdl.gbras_bench import run

benchmark = run(settings, Path(settings.output_root) / settings.run_directory_name,
                smoke=os.environ.get("CVDL_BENCH_SMOKE") == "1", started_at=BENCH_STARTED)
print("BENCHMARK, NOT A RESULT")
print(json.dumps({key: benchmark[key] for key in ("state", "winner", "projection")}, indent=2))
print("Full evidence: benchmark-summary.json, numerics.json and per-trial files.")
if benchmark["state"] not in {"complete", "smoke-complete", "budget_exhausted"}:
    raise RuntimeError("Benchmark failed; outputs preserved: " + benchmark["state"])
